# KSE-30 Evaluation, Results & Analysis
### Notebook 6 — All Final Figures + Assignment Tables
---
**Upload these 6 files (all from Notebook 5 outputs):**
1. `model_metrics_summary.csv`
2. `per_stock_metrics.csv`
3. `KSE30_portfolio_weights.csv`
4. `KSE30_portfolio_performance.csv`
5. `KSE30_daily_port_returns.csv`
6. `KSE30_bl_returns.csv`

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy -q
print('Done.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import Patch
import glob, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi']    = 300
plt.rcParams['savefig.dpi']   = 300
plt.rcParams['axes.grid']     = True
plt.rcParams['grid.alpha']    = 0.3
plt.rcParams['font.size']     = 9
sns.set_style('whitegrid')

RISK_FREE_RATE = 0.20
TRADING_DAYS   = 252
print('Ready.')

---
## 1. Load All Files

In [ ]:
from google.colab import files
print('Upload all 6 files (see notebook title for list)')
uploaded = files.upload()
fnames   = list(uploaded.keys())

def find(kw):
    for f in fnames:
        if kw.lower() in f.lower(): return f
    return None

model_metrics = pd.read_csv(find('model_metrics'))
per_stock     = pd.read_csv(find('per_stock'))
weights       = pd.read_csv(find('weights'))
performance   = pd.read_csv(find('performance'))
daily_ret     = pd.read_csv(find('daily_port'))
bl_returns    = pd.read_csv(find('bl_returns'))

daily_ret['Date'] = pd.to_datetime(daily_ret['Date'])

print('Loaded:')
for name, df in [('model_metrics',model_metrics),('per_stock',per_stock),
                  ('weights',weights),('performance',performance),
                  ('daily_ret',daily_ret),('bl_returns',bl_returns)]:
    print(f'  {name:<20} {df.shape}')

---
## 2. ML Model Evaluation Figures

In [ ]:
# Figure 1 — Model Metrics Heatmap
p_da   = model_metrics.pivot(index='Model',columns='Split',values='DA_%')[['Train','Val','Test']]
p_rmse = model_metrics.pivot(index='Model',columns='Split',values='RMSE')[['Train','Val','Test']]
p_r2   = model_metrics.pivot(index='Model',columns='Split',values='R2')[['Train','Val','Test']]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.heatmap(p_da,   annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=40, vmax=60, ax=axes[0], linewidths=0.5, annot_kws={'size':12})
axes[0].set_title('Directional Accuracy (%)', fontweight='bold')
sns.heatmap(p_rmse, annot=True, fmt='.5f', cmap='RdYlGn_r',
            ax=axes[1], linewidths=0.5, annot_kws={'size':12})
axes[1].set_title('RMSE', fontweight='bold')
sns.heatmap(p_r2,   annot=True, fmt='.4f', cmap='RdYlGn',
            vmin=-1, vmax=0.5, ax=axes[2], linewidths=0.5, annot_kws={'size':12})
axes[2].set_title('R² Score', fontweight='bold')
plt.suptitle('ML Model Performance — Train / Val / Test', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eval_01_ModelHeatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('Saved: fig_eval_01_ModelHeatmap.png')

In [ ]:
# Figure 2 — Per-Stock DA Distribution
ps = per_stock.sort_values('DA_%', ascending=False)
da_colors = ['#2ecc71' if v>=55 else '#e67e22' if v>=50 else '#e74c3c' for v in ps['DA_%']]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
axes[0].bar(ps['Ticker'].str.replace('.KA',''), ps['DA_%'],
            color=da_colors, edgecolor='white')
axes[0].axhline(50, color='black', ls='--', lw=1.2, label='Random (50%)')
axes[0].axhline(ps['DA_%'].mean(), color='blue', ls='-.', lw=1,
                label=f'Mean = {ps["DA_%"].mean():.1f}%')
axes[0].set_title('Hybrid — Directional Accuracy per Stock (Test)', fontweight='bold')
axes[0].set_ylabel('DA (%)')
axes[0].tick_params(axis='x', rotation=90, labelsize=7)
axes[0].legend(handles=[
    Patch(facecolor='#2ecc71',label='≥55%'),
    Patch(facecolor='#e67e22',label='50–55%'),
    Patch(facecolor='#e74c3c',label='<50%'),
], fontsize=8)

axes[1].hist(ps['DA_%'], bins=10, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[1].axvline(50, color='red',  ls='--', lw=1.2, label='Random (50%)')
axes[1].axvline(ps['DA_%'].mean(), color='blue', ls='-.', lw=1,
                label=f'Mean = {ps["DA_%"].mean():.1f}%')
axes[1].set_title('Distribution of Per-Stock DA', fontweight='bold')
axes[1].set_xlabel('DA (%)')
axes[1].set_ylabel('Stock Count')
axes[1].legend(fontsize=8)

plt.suptitle('Per-Stock Prediction Accuracy — Hybrid Model',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eval_02_PerStockDA.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('Saved: fig_eval_02_PerStockDA.png')

In [ ]:
# Figure 3 — Model vs Baselines
test_m   = model_metrics[model_metrics['Split']=='Test'].set_index('Model')
models   = ['Random\n(50%)', 'XGBoost', 'LSTM', 'Hybrid']
da_vals  = [50.0, test_m.loc['XGBoost','DA_%'], test_m.loc['LSTM','DA_%'], test_m.loc['Hybrid','DA_%']]
rmse_v   = [None, test_m.loc['XGBoost','RMSE'], test_m.loc['LSTM','RMSE'], test_m.loc['Hybrid','RMSE']]
clrs     = ['#bdc3c7','#3498db','#e74c3c','#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bars = axes[0].bar(models, da_vals, color=clrs, edgecolor='white', width=0.5)
axes[0].axhline(50, color='black', ls='--', lw=1, label='Random baseline')
axes[0].set_title('Directional Accuracy (%) vs Baselines', fontweight='bold')
axes[0].set_ylabel('DA (%)')
axes[0].set_ylim(44, 55)
axes[0].legend(fontsize=8)
for bar, v in zip(bars, da_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.1,
                 f'{v:.2f}%', ha='center', fontsize=8, fontweight='bold')

rmse_models = ['XGBoost','LSTM','Hybrid']
rmse_vals   = [test_m.loc[m,'RMSE'] for m in rmse_models]
bars2 = axes[1].bar(rmse_models, rmse_vals, color=['#3498db','#e74c3c','#2ecc71'],
                    edgecolor='white', width=0.5)
axes[1].set_title('RMSE Comparison — Test Set', fontweight='bold')
axes[1].set_ylabel('RMSE')
for bar, v in zip(bars2, rmse_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, v*1.005,
                 f'{v:.5f}', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Model Performance vs Baseline Methods', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eval_03_BaselineComparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('Saved: fig_eval_03_BaselineComparison.png')

---
## 3. Portfolio Evaluation Figures

In [ ]:
# Figure 4 — Portfolio Weights + Sector
wdf  = weights.sort_values('BL_Weight', ascending=False)
top15= wdf.head(15)
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

clrs = plt.cm.RdYlGn(np.linspace(0.2,0.9,len(top15)))
axes[0].bar(top15['Ticker'].str.replace('.KA',''), top15['BL_Weight']*100,
            color=clrs[::-1], edgecolor='white')
axes[0].axhline(8.0, color='red', ls='--', lw=1, label='8% cap')
axes[0].axhline(100/len(wdf), color='blue', ls='-.', lw=1,
                label=f'Equal ({100/len(wdf):.1f}%)')
axes[0].set_title('Top 15 Holdings — BL Portfolio (%)', fontweight='bold')
axes[0].set_ylabel('Weight (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=8)
for i, (_, row) in enumerate(top15.iterrows()):
    axes[0].text(i, row['BL_Weight']*100+0.1,
                 f'{row["BL_Weight"]*100:.1f}%', ha='center', fontsize=7)

sector_w = weights.groupby('Sector')['BL_Weight'].sum().sort_values(ascending=True)
clrs2    = plt.cm.Set2(np.linspace(0,1,len(sector_w)))
axes[1].barh(sector_w.index, sector_w.values*100, color=clrs2, edgecolor='white')
axes[1].set_title('Sector Allocation — BL Portfolio (%)', fontweight='bold')
axes[1].set_xlabel('Allocation (%)')
for i, v in enumerate(sector_w.values*100):
    axes[1].text(v+0.1, i, f'{v:.1f}%', va='center', fontsize=8)

plt.suptitle('BL Portfolio — Weight Distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eval_04_PortWeights.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('Saved: fig_eval_04_PortWeights.png')

In [ ]:
# Figure 5 — Cumulative Returns + Drawdown Combined
def drawdown(r):
    c = np.exp(np.cumsum(r))
    return (c - np.maximum.accumulate(c)) / np.maximum.accumulate(c)

bl_cum  = np.exp(np.cumsum(daily_ret['BL_Portfolio'])) - 1
eq_cum  = np.exp(np.cumsum(daily_ret['Equal_Weight'])) - 1
eqo_cum = np.exp(np.cumsum(daily_ret['Equilibrium']))  - 1

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 1, height_ratios=[2,1], hspace=0.05)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

ax1.plot(daily_ret['Date'], bl_cum*100,  color='#2ecc71', lw=1.5, label='BL Portfolio (ML Views)')
ax1.plot(daily_ret['Date'], eq_cum*100,  color='#3498db', lw=1.2, ls='--', label='Equal Weight')
ax1.plot(daily_ret['Date'], eqo_cum*100, color='#e74c3c', lw=1.2, ls=':',  label='Equilibrium Only')
ax1.axhline(0, color='black', lw=0.5)
ax1.set_ylabel('Cumulative Return (%)')
ax1.set_title('Portfolio Performance — Test Period (Jul 2024 – Dec 2025)',
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
plt.setp(ax1.get_xticklabels(), visible=False)

ax2.fill_between(daily_ret['Date'], drawdown(daily_ret['BL_Portfolio'])*100,
                 0, alpha=0.5, color='#2ecc71', label='BL Drawdown')
ax2.fill_between(daily_ret['Date'], drawdown(daily_ret['Equal_Weight'])*100,
                 0, alpha=0.3, color='#3498db', label='EW Drawdown')
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.tick_params(axis='x', rotation=30)
ax2.legend(fontsize=8)

plt.savefig('fig_eval_05_CumReturn_Drawdown.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('Saved: fig_eval_05_CumReturn_Drawdown.png')

In [ ]:
# Figure 6 — Portfolio Metrics Bar
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
clrs = ['#2ecc71','#3498db','#e74c3c']
for ax, col, title in zip(axes,
    ['Ann_Return','Ann_Vol','Sharpe','Max_DD'],
    ['Ann. Return (%)','Ann. Volatility (%)','Sharpe Ratio','Max Drawdown (%)']):
    vals = performance[col].values
    if col in ['Ann_Return','Ann_Vol','Max_DD']: vals = vals * 100
    bars = ax.bar(performance['Portfolio'], vals, color=clrs, edgecolor='white', width=0.5)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=20, labelsize=7)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height() + abs(bar.get_height())*0.02,
                f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')
plt.suptitle('Portfolio Performance Metrics — Test Period', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eval_06_PortMetrics.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('Saved: fig_eval_06_PortMetrics.png')

In [ ]:
# Figure 7 — Monthly Returns Heatmap
dr = daily_ret.copy()
dr['Month'] = dr['Date'].dt.to_period('M')
monthly_bl  = dr.groupby('Month')['BL_Portfolio'].sum() * 100
m_df        = monthly_bl.reset_index()
m_df['Year'] = m_df['Month'].dt.year
m_df['Mon']  = m_df['Month'].dt.month
pivot = m_df.pivot(index='Year', columns='Mon', values='BL_Portfolio')
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
pivot.columns = [month_names[i-1] for i in pivot.columns]

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size':9})
ax.set_title('BL Portfolio — Monthly Returns Heatmap (%)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eval_07_MonthlyReturns.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('Saved: fig_eval_07_MonthlyReturns.png')

In [ ]:
# Figure 8 — Rolling Sharpe + Volatility
def roll_sharpe(r, w=30):
    s = pd.Series(r.values, index=daily_ret['Date'])
    return (s.rolling(w).mean()-RISK_FREE_RATE/TRADING_DAYS)/s.rolling(w).std()*np.sqrt(TRADING_DAYS)

def roll_vol(r, w=30):
    s = pd.Series(r.values, index=daily_ret['Date'])
    return s.rolling(w).std() * np.sqrt(TRADING_DAYS) * 100

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
axes[0].plot(daily_ret['Date'], roll_sharpe(daily_ret['BL_Portfolio']),
             color='#2ecc71', lw=1, label='BL Portfolio')
axes[0].plot(daily_ret['Date'], roll_sharpe(daily_ret['Equal_Weight']),
             color='#3498db', lw=1, ls='--', label='Equal Weight')
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_title('Rolling 30-Day Sharpe Ratio', fontweight='bold')
axes[0].set_ylabel('Sharpe Ratio'); axes[0].legend(fontsize=8)

axes[1].plot(daily_ret['Date'], roll_vol(daily_ret['BL_Portfolio']),
             color='#2ecc71', lw=1, label='BL Portfolio')
axes[1].plot(daily_ret['Date'], roll_vol(daily_ret['Equal_Weight']),
             color='#3498db', lw=1, ls='--', label='Equal Weight')
axes[1].set_title('Rolling 30-Day Annualised Volatility (%)', fontweight='bold')
axes[1].set_ylabel('Volatility (%)')
axes[1].set_xlabel('Date')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=8)

plt.suptitle('Rolling Risk Metrics — BL Portfolio vs Equal Weight',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_eval_08_RollingMetrics.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('Saved: fig_eval_08_RollingMetrics.png')

---
## 4. Final Summary Tables

In [ ]:
print('TABLE 1: ML MODEL PERFORMANCE')
print(model_metrics.to_string(index=False))
print()
print('TABLE 2: PORTFOLIO PERFORMANCE')
print(performance.round(4).to_string(index=False))
print()
print('TABLE 3: TOP 10 HOLDINGS')
print(weights.sort_values('BL_Weight',ascending=False).head(10)[['Ticker','Sector','BL_Weight']].round(4).to_string(index=False))
print()
print('TABLE 4: SECTOR ALLOCATION')
sa = weights.groupby('Sector')['BL_Weight'].sum().sort_values(ascending=False).reset_index()
sa['Pct'] = (sa['BL_Weight']*100).round(2)
print(sa[['Sector','Pct']].to_string(index=False))
print()
print('TABLE 5: PER STOCK DA')
print(per_stock.to_string(index=False))

In [ ]:
# Key findings — copy this into your Results Analysis section
bl  = performance[performance['Portfolio'].str.contains('BL')].iloc[0]
eq  = performance[performance['Portfolio'].str.contains('Equal')].iloc[0]
t   = model_metrics[model_metrics['Split']=='Test']

print('KEY FINDINGS FOR RESULTS ANALYSIS SECTION')
print('='*60)
print(f'ML Prediction:')
print(f'  XGBoost  DA = {t[t["Model"]=="XGBoost"]["DA_%"].values[0]:.2f}%')
print(f'  LSTM     DA = {t[t["Model"]=="LSTM"]["DA_%"].values[0]:.2f}%')
print(f'  Hybrid   DA = {t[t["Model"]=="Hybrid"]["DA_%"].values[0]:.2f}%')
print(f'  Random baseline = 50%')
print(f'  → DA below 50% is consistent with EMH in emerging markets')
print()
print(f'Portfolio:')
print(f'  BL Ann Return  = {bl["Ann_Return"]*100:.2f}%')
print(f'  BL Sharpe      = {bl["Sharpe"]:.4f}')
print(f'  BL Max DD      = {bl["Max_DD"]*100:.2f}%')
print(f'  EW Sharpe      = {eq["Sharpe"]:.4f}')
print(f'  Sharpe diff    = {bl["Sharpe"]-eq["Sharpe"]:+.4f}')
print()
print(f'Concentration:')
print(f'  Max holding    = {weights["BL_Weight"].max()*100:.2f}% (cap = 8%)')
print(f'  Stocks at cap  = {(weights["BL_Weight"] >= 0.075).sum()}')
print(f'  Sectors covered= {weights["Sector"].nunique()}')
print(f'  Effective N    = {1/np.sum(weights["BL_Weight"]**2):.1f} stocks')

In [ ]:
# Save summary tables
model_metrics.to_csv('final_model_metrics.csv', index=False)
performance.to_csv('final_portfolio_performance.csv', index=False)
weights[['Ticker','Sector','BL_Weight']].sort_values(
    'BL_Weight', ascending=False).to_csv('final_portfolio_weights.csv', index=False)
sa.to_csv('final_sector_allocation.csv', index=False)
per_stock.to_csv('final_per_stock_metrics.csv', index=False)

from google.colab import files
import glob

print('Downloading all figures (300 DPI)...')
for f in sorted(glob.glob('fig_eval_*.png')):
    files.download(f); print(f'  ✓ {f}')

print('\nDownloading summary tables...')
for f in ['final_model_metrics.csv','final_portfolio_performance.csv',
          'final_portfolio_weights.csv','final_sector_allocation.csv',
          'final_per_stock_metrics.csv']:
    files.download(f); print(f'  ✓ {f}')

print('\nAll done! You now have everything for the assignment report.')

---
## ✅ Complete — 36 Figures Total
| Notebook | Figures |
|---|---|
| Feature Engineering | fig01–fig11 |
| Model | fig_model_01–10 |
| Portfolio | fig_port_01–07 |
| Evaluation | fig_eval_01–08 |
| **Total** | **36 figures @ 300 DPI** |